# ChemIntel: ChemPile retrieval + ChemBench evaluation (Colab)
**Status: this notebook has NOT been run on a GPU by the author.** Run the cells in order. Section 3 (language-model evaluation) and
section 4 (optional training) are the parts that need a GPU runtime (Runtime > Change runtime type > GPU). Colab GPU availability and limits
are not guaranteed (see the Colab FAQ). Fill in only measured numbers in your report.

## 0. Setup
Upload `chemintel_llm.zip` to the Colab files panel first.

In [ ]:
!unzip -q -o chemintel_llm.zip
%cd chemintel_llm
!pip -q install torch transformers peft accelerate datasets bitsandbytes streamlit
import torch; print('GPU:', torch.cuda.is_available() and torch.cuda.get_device_name(0))

## 1. Data (streams ChemPile-Education, removes ChemBench overlap) and BM25 index
Raise `--max-train-docs` for a bigger corpus (more RAM and time). 3000 docs is about 5M tokens.

In [ ]:
!python scripts/01_prepare_corpus.py --out data --max-train-docs 3000
!python scripts/02_build_index.py

## 2. Model-free baselines (CPU, minutes)
These use no language model. They are reference lines, not LLM results.

In [ ]:
!python scripts/05_eval_chembench.py --backend chance --limit 500 --name chance_500
!python scripts/05_eval_chembench.py --backend retrieval_only --limit 500 --name retrieval_only_500

## 3. Language model, with and without retrieval (needs GPU for speed)
Base model default `HuggingFaceTB/SmolLM2-1.7B-Instruct` (Apache-2.0 per its Hugging Face licence file; verify). Use `--limit 500` first to time it.

In [ ]:
MODEL = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
!python scripts/05_eval_chembench.py --backend hf --model $MODEL --limit 500 --name base_noRAG
!python scripts/05_eval_chembench.py --backend hf --model $MODEL --limit 500 --rag --name base_RAG

## 4. OPTIONAL: QLoRA continued pretraining on the ChemPile text, then re-evaluate
Small budget first (`--max-steps 50`) to see speed. A short run on a few million tokens may not change ChemBench accuracy; report whatever you measure.

In [ ]:
!python scripts/03_train_qlora.py --model $MODEL --data data/train.jsonl --out runs/cpt --max-tokens 5000000 --max-steps 100
!python scripts/05_eval_chembench.py --backend hf --model $MODEL --adapter runs/cpt --limit 500 --name cpt_noRAG
!python scripts/05_eval_chembench.py --backend hf --model $MODEL --adapter runs/cpt --limit 500 --rag --name cpt_RAG

## 5. Results table (only measured runs appear)

In [ ]:
!python scripts/06_report.py results